# CS781 Capstone Phase 5: Legal Statute Identification (LSI)
## Variant C: Hybrid Neural-Citation Branch (Whole-Document Preprocessing)
### Architectural Profile: Model 4 Peak Capacity Boundary Decoder (153.9M Parameters)
* **Backbone:** 14 layers, d=896, H_q=14, H_kv=7, GQA 2:1 (100% UNFROZEN)
* **Whole-Doc Citation Scanner:** CPU regex feature scanner across 100% of raw text (sec_N, act_name, sec_N@act_X within 80 chars, length buckets). Eliminates long-document truncation.
* **Citation Branch:** EmbeddingBag(V=30k, 256) -> LayerNorm -> GELU -> Dropout(0.2) -> Linear(256 -> 100) yielding z_cite.
* **Gated Residual Fusion:** z_final = z_neural + g * z_cite (learnable gate g initialized to 1.0).
* **Citation Dropout:** 15% random dropout of citation features during training so neural path keeps learning.
* **Loss:** ASL(z_final) + 0.3 * ASL(z_cite) + 0.3 * ASL(z_neural) evaluated strictly in FP32.
* **Evaluation:** Split-half threshold calibration (dev_tune -> dev_eval), logging Macro-F1, Macro-AP, Micro-F1, and bucketed F1.


In [1]:
# 1. Environment & Self-Contained Phase 5 Core Library (100% Standalone)
import os
import sys
import gc
import json
import time
import math
import random
import zipfile
import shutil
import base64
import zlib
import types
import threading
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Any, Optional, Union

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ==============================================================================
# SELF-CONTAINED PHASE 5 CORE MODULE INLINING (Zero external file dependencies)
# ==============================================================================
_PHASE5_CORE_B64 = '...'

try:
    _core_src = zlib.decompress(base64.b64decode(_PHASE5_CORE_B64)).decode("utf-8")
    # 1. Write phase5_common.py directly to working directory so any subprocess or script can access it
    with open("phase5_common.py", "w", encoding="utf-8") as _f:
        _f.write(_core_src)
    
    # 2. Execute directly into an in-memory module
    p5 = types.ModuleType("phase5_common")
    exec(_core_src, p5.__dict__)
    sys.modules["phase5_common"] = p5
    print("Self-contained phase5_common core initialized successfully (100% standalone).")
except Exception as _e:
    print(f"Warning during self-contained core unpacking: {_e}")
    import phase5_common as p5

# Secure dynamic token retrieval (never hardcoded)
HF_TOKEN = p5.get_hf_token() or "YOUR_HF_TOKEN"
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
if HF_TOKEN:
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN, add_to_git_credential=False)
    except Exception:
        pass

if HF_TOKEN:
    print("Hugging Face Hub authentication token identified successfully.")
else:
    print("WARNING: HF_TOKEN not found. Remote Hub sync will be local only.")


Self-contained phase5_common core initialized successfully (100% standalone).


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face Hub authentication token identified successfully.


In [2]:
# 2. Compute Device Configuration & Reproducibility
device = p5.get_best_device()
print(f"Active Compute Device: {device}")

p5.seed_everything(42)
print("Deterministic random seeds initialized (Seed 42).")

# Enable memory-efficient SDPA attention if on CUDA
if device.type == "cuda":
    torch.backends.cuda.enable_mem_efficient_sdp(True)
    torch.backends.cuda.enable_math_sdp(True)
    torch.backends.cuda.enable_flash_sdp(False)
    print("PyTorch SDPA memory-efficient attention enabled.")


Active Compute Device: cuda
Deterministic random seeds initialized (Seed 42).
PyTorch SDPA memory-efficient attention enabled.


In [ ]:
# 3. Experiment Configuration & Hyperparameters
PROFILE = "core"  # Options: "core" | "smoke"
EPOCHS = 1 if PROFILE == "smoke" else 1
HUB_SYNC_STEP_INTERVAL = 200
PAD_TOKEN_ID = p5.PAD_TOKEN_ID

EXPERIMENT_NAME = "sub01_variant_c_citation"
MODEL_KEY = "model4"
SUBFOLDER = "model4-capacity-boundary-154m"
TARGET_METRIC = "Macro-F1 (Official Primary Metric)"

OUTPUT_DIR = os.path.abspath(f"./checkpoint_{EXPERIMENT_NAME}")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")
PREDICTION_JSON = os.path.join(OUTPUT_DIR, "prediction.json")
METRICS_JSON = os.path.join(OUTPUT_DIR, "val_metrics.json")
LOSS_PLOT_PATH = os.path.join(OUTPUT_DIR, "loss_plot.png")

os.makedirs(OUTPUT_DIR, exist_ok=True)

MICRO_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8  # Effective batch size = 16
LR_BACKBONE = 3.0e-5
LR_HEAD = 8.0e-4
LR_CITE = 1.0e-3
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
MAX_GRAD_NORM = 1.0
D_ATTN = 128
CHUNK_WINDOW = 512
CHUNK_OVERLAP = 64
CHUNK_STRIDE = 448
TRAIN_SAMPLE_K = 6
CITE_DROPOUT = 0.15
CITE_AUX_WEIGHT = 0.3

print(f"Configuration set for: {EXPERIMENT_NAME}")
print(f"Run Profile: {PROFILE.upper()} | Configured Epochs: {EPOCHS} | Sync Interval: {HUB_SYNC_STEP_INTERVAL} steps")
print(f"Output directory: {OUTPUT_DIR}")


In [4]:
# 4. Ingest Phase 5 Dataset & Setup Tokenizer
print("Ingesting Phase 5 LSI dataset...")
train_records, dev_tune, dev_eval, test_records, statute2idx, idx2statute = p5.ingest_phase5_dataset(HF_TOKEN)

# Handle SMOKE test profile slicing
if PROFILE == "smoke":
    print("=" * 65)
    print(">>> [PROFILE: SMOKE] Slicing dataset for rapid end-to-end verification <<<")
    print(f">>> Configured EPOCHS: {EPOCHS} | Hub Sync Interval: {HUB_SYNC_STEP_INTERVAL} steps")
    print("=" * 65)
    train_records = train_records[:32]
    dev_tune = dev_tune[:16]
    dev_eval = dev_eval[:16]
    test_records = test_records[:16]
    if EPOCHS > 2:
        print(f"[Smoke Mode Notice] Capping EPOCHS from {EPOCHS} to 1 for quick execution.")
        EPOCHS = 1

NUM_LABELS = len(statute2idx)
print(f"Total Unique Statutes (|L|): {NUM_LABELS}")
print(f"Train documents: {len(train_records):,}")
print(f"Dev Tune documents: {len(dev_tune):,}")
print(f"Dev Eval documents: {len(dev_eval):,}")
print(f"Test documents: {len(test_records):,}")

# Load custom scratch-trained 32K legal tokenizer
tokenizer = p5.load_legal_tokenizer(p5.HF_TOKENIZER_REPO, token=HF_TOKEN)
p5.PAD_TOKEN_ID = tokenizer.pad_token_id
PAD_TOKEN_ID = tokenizer.pad_token_id
if tokenizer.eos_token_id is not None:
    p5.EOS_TOKEN_ID = tokenizer.eos_token_id
    EOS_TOKEN_ID = tokenizer.eos_token_id
print(f"Loaded Tokenizer: Vocab={len(tokenizer):,}, Pad={tokenizer.pad_token_id}, EOS={tokenizer.eos_token_id}")


Ingesting Phase 5 LSI dataset...


Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

[DataIngest] Successfully loaded 42,750 train, 10,181 dev, 13,019 test records.
[DataIngest] Total Unique Statute Labels (|L|): 100
[DataIngest] Split-Half Validation: dev_tune=5,090 docs, dev_eval=5,091 docs.
Total Unique Statutes (|L|): 100
Train documents: 42,750
Dev Tune documents: 5,090
Dev Eval documents: 5,091
Test documents: 13,019


tokenizer_config.json:   0%|          | 0.00/324 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.29M [00:00<?, ?B/s]

Loaded Tokenizer: Vocab=32,000, Pad=0, EOS=0


In [ ]:
# 5. Strict Pretrained Backbone Loading & Variant C Citation Branch
print(f"Loading Model 4 backbone from {p5.HF_MASTER_REPO}/{SUBFOLDER}...")
backbone, config_dict = p5.load_strict_phase1_backbone(MODEL_KEY, hf_token=HF_TOKEN)

for param in backbone.parameters():
    param.requires_grad = True

import re
from collections import Counter

# Quick Statutory Citation Feature Extraction
SECTION_REGEX = re.compile(r'\b(?:section|sec\.?|s\.)\s*(\d+[a-z]?)\b', re.IGNORECASE)
ACT_REGEX = re.compile(r'\b(ipc|crpc|cpc|constitution|nia|ndps|evidence|motor vehicles?|arbitration)\b', re.IGNORECASE)

def extract_doc_citations(text):
    text_lower = text.lower()
    features = []
    secs = SECTION_REGEX.findall(text_lower)
    acts = ACT_REGEX.findall(text_lower)
    for s in secs[:30]:
        features.append(f"sec_{s}")
    for a in acts[:20]:
        features.append(f"act_{a}")
    for m in re.finditer(r'\b(?:section|sec\.?|s\.)\s*(\d+[a-z]?)\b.{0,80}?\b(ipc|crpc|cpc|constitution|nia|ndps)\b', text_lower):
        features.append(f"sec_{m.group(1)}@{m.group(2)}")
    doc_len = len(text_lower.split())
    len_bucket = min(10, doc_len // 1000)
    features.append(f"len_bucket_{len_bucket}")
    return features

# Build vocabulary on train records
vocab_counter = Counter()
for r in train_records:
    vocab_counter.update(extract_doc_citations(r["text"]))

min_df = 2 if PROFILE == "smoke" else 3
cite_vocab = {f: idx + 1 for idx, (f, c) in enumerate(vocab_counter.most_common(30000)) if c >= min_df}
CITE_VOCAB_SIZE = len(cite_vocab) + 2
print(f"Built Whole-Document Citation Vocabulary: {len(cite_vocab):,} features (Total Vocab Space: {CITE_VOCAB_SIZE})")

class WholeDocCitationBranch(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int = 256, num_labels: int = 100):
        super().__init__()
        self.embedding = nn.EmbeddingBag(vocab_size, embed_dim, mode="sum")
        self.norm = nn.LayerNorm(embed_dim)
        self.gelu = nn.GELU()
        self.dropout = nn.Dropout(0.2)
        self.proj = nn.Linear(embed_dim, num_labels)

    def forward(self, input_indices: torch.Tensor, offsets: torch.Tensor, per_sample_weights: Optional[torch.Tensor] = None) -> torch.Tensor:
        h = self.embedding(input_indices, offsets, per_sample_weights=per_sample_weights)
        h = self.proj(self.dropout(self.gelu(self.norm(h))))
        return h

class Phase5VariantCModel(nn.Module):
    def __init__(self, backbone, hidden_size: int, num_labels: int, cite_vocab_size: int, d_attn: int = 128):
        super().__init__()
        self.backbone = backbone
        self.hidden_size = hidden_size
        self.num_labels = num_labels
        self.d_chunk_feat = 2 * hidden_size

        self.layer_mixture = p5.Top4LayerMixture()
        self.chunk_attention = p5.LabelWiseChunkAttention(
            d_chunk_feat=self.d_chunk_feat,
            num_labels=num_labels,
            d_attn=d_attn
        )
        self.classifier_head = p5.StatuteClassifierHead(
            d_chunk_feat=self.d_chunk_feat,
            num_labels=num_labels
        )
        self.citation_branch = WholeDocCitationBranch(cite_vocab_size, 256, num_labels)
        self.gate = nn.Parameter(torch.ones(num_labels))

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        chunk_mask: Optional[torch.Tensor] = None,
        cite_indices: Optional[torch.Tensor] = None,
        cite_offsets: Optional[torch.Tensor] = None
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        B, M, S = input_ids.size()
        total_chunks = B * M
        flat_input_ids = input_ids.view(total_chunks, S)
        flat_attn_mask = attention_mask.view(total_chunks, S)

        # Valid-chunks-only forward filtering
        if chunk_mask is not None:
            flat_cmask = chunk_mask.view(-1)
            valid_idx = torch.nonzero(flat_cmask > 0, as_tuple=True)[0]
        else:
            valid_idx = torch.arange(total_chunks, device=input_ids.device)

        if len(valid_idx) == 0:
            zero_z = torch.zeros(B, self.num_labels, device=input_ids.device)
            return zero_z, zero_z, zero_z

        v_ids = flat_input_ids[valid_idx]
        v_mask = flat_attn_mask[valid_idx]

        chunk_batch_size = 16
        v_feats_list = []
        for c_start in range(0, len(valid_idx), chunk_batch_size):
            c_end = min(c_start + chunk_batch_size, len(valid_idx))
            sub_ids = v_ids[c_start:c_end]
            sub_mask = v_mask[c_start:c_end]

            outputs = self.backbone(
                input_ids=sub_ids,
                attention_mask=sub_mask,
                output_hidden_states=True,
                return_dict=True
            )
            top4 = list(outputs.hidden_states[-4:])
            mixed_hidden = self.layer_mixture(top4)
            b_feats = p5.extract_chunk_features(mixed_hidden, sub_mask)
            v_feats_list.append(b_feats)

        v_feats = torch.cat(v_feats_list, dim=0)

        chunk_feats = torch.zeros(total_chunks, self.d_chunk_feat, dtype=v_feats.dtype, device=v_feats.device)
        chunk_feats[valid_idx] = v_feats
        chunk_feats = chunk_feats.view(B, M, self.d_chunk_feat)

        statute_reps = self.chunk_attention(chunk_feats, chunk_mask=chunk_mask)
        z_neural = self.classifier_head(statute_reps)

        # Citation Branch
        if cite_indices is not None and cite_offsets is not None:
            z_cite = self.citation_branch(cite_indices, cite_offsets)
        else:
            z_cite = torch.zeros_like(z_neural)

        z_final = z_neural + self.gate.unsqueeze(0) * z_cite
        return z_final, z_cite, z_neural

model = Phase5VariantCModel(backbone, config_dict["hidden_size"], NUM_LABELS, CITE_VOCAB_SIZE)
raw_model = model
if torch.cuda.device_count() > 1:
    print(f"Enabling DataParallel across {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(raw_model)
model = model.to(device)

total_params = sum(p.numel() for p in raw_model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model 4 Total Architecture Parameters: {total_params / 1e6:.2f}M")
print(f"Model 4 Trainable Parameters: {trainable_params / 1e6:.2f}M (100% UNFROZEN)")
assert total_params == trainable_params, "All parameters must be unfrozen for Full Fine-Tuning!"


In [ ]:
# 6. Document Dataset with Whole-Document Citation Extraction
class DocumentLSIDatasetVariantC(Dataset):
    def __init__(self, records, tokenizer, is_train=False):
        self.records = records
        self.tokenizer = tokenizer
        self.is_train = is_train

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        tokens = self.tokenizer.encode(rec["text"], add_special_tokens=False) + [self.tokenizer.eos_token_id]
        input_ids, attn_mask = p5.chunk_document_tokens(tokens, window_size=CHUNK_WINDOW, stride=CHUNK_STRIDE)

        if self.is_train:
            input_ids, attn_mask = p5.sample_training_chunks(
                input_ids, attn_mask, rec["text"], self.tokenizer, k=TRAIN_SAMPLE_K
            )

        label_vec = torch.zeros(NUM_LABELS, dtype=torch.float32)
        for lbl in rec.get("labels", []):
            if int(lbl) in statute2idx:
                label_vec[statute2idx[int(lbl)]] = 1.0

        # Whole-document citation feature IDs
        feats = extract_doc_citations(rec["text"])
        feat_ids = [cite_vocab[f] for f in feats if f in cite_vocab]
        if not feat_ids:
            feat_ids = [0]

        return input_ids, attn_mask, label_vec, rec["id"], feat_ids

def collate_doc_batch(batch):
    max_m = max(item[0].size(0) for item in batch)
    s_len = batch[0][0].size(1)
    b_size = len(batch)

    padded_ids = torch.full((b_size, max_m, s_len), p5.PAD_TOKEN_ID, dtype=torch.long)
    padded_masks = torch.zeros((b_size, max_m, s_len), dtype=torch.long)
    chunk_mask = torch.zeros((b_size, max_m), dtype=torch.float32)
    targets = torch.stack([item[2] for item in batch], dim=0)
    doc_ids = [item[3] for item in batch]

    for i, item in enumerate(batch):
        m_len = item[0].size(0)
        padded_ids[i, :m_len, :] = item[0]
        padded_masks[i, :m_len, :] = item[1]
        chunk_mask[i, :m_len] = 1.0

    # Flat offsets for EmbeddingBag
    flat_indices = []
    offsets = []
    curr = 0
    for item in batch:
        offsets.append(curr)
        f_ids = item[4]
        flat_indices.extend(f_ids)
        curr += len(f_ids)

    indices_tensor = torch.tensor(flat_indices, dtype=torch.long)
    offsets_tensor = torch.tensor(offsets, dtype=torch.long)

    return padded_ids, padded_masks, chunk_mask, targets, doc_ids, indices_tensor, offsets_tensor

train_dataset = DocumentLSIDatasetVariantC(train_records, tokenizer, is_train=True)
dev_tune_dataset = DocumentLSIDatasetVariantC(dev_tune, tokenizer, is_train=False)
dev_eval_dataset = DocumentLSIDatasetVariantC(dev_eval, tokenizer, is_train=False)
test_dataset = DocumentLSIDatasetVariantC(test_records, tokenizer, is_train=False)

train_loader = DataLoader(train_dataset, batch_size=MICRO_BATCH_SIZE, shuffle=True, collate_fn=collate_doc_batch)
dev_tune_loader = DataLoader(dev_tune_dataset, batch_size=MICRO_BATCH_SIZE, shuffle=False, collate_fn=collate_doc_batch)
dev_eval_loader = DataLoader(dev_eval_dataset, batch_size=MICRO_BATCH_SIZE, shuffle=False, collate_fn=collate_doc_batch)
test_loader = DataLoader(test_dataset, batch_size=MICRO_BATCH_SIZE, shuffle=False, collate_fn=collate_doc_batch)

print("DataLoaders initialized successfully with Variant C citation batching.")


In [ ]:
# 7. Pre-Flight Verification & Tri-Head Gradient Flow Assertions
print("Running mandatory 1-batch pre-flight gradient flow verification (Variant C)...")
model.train()

test_batch = next(iter(train_loader))
b_ids, b_masks, b_cmask, b_targets, _, b_cindices, b_coffsets = test_batch

b_ids = b_ids.to(device)
b_masks = b_masks.to(device)
b_cmask = b_cmask.to(device)
b_targets = b_targets.to(device)
b_cindices = b_cindices.to(device)
b_coffsets = b_coffsets.to(device)

criterion = p5.AsymmetricLoss(gamma_neg=4.0, gamma_pos=0.0, clip_margin=0.05)

z_final, z_cite, z_neural = model(b_ids, b_masks, chunk_mask=b_cmask, cite_indices=b_cindices, cite_offsets=b_coffsets)
test_loss = criterion(z_final.float(), b_targets.float()) + 0.3 * criterion(z_cite.float(), b_targets.float()) + 0.3 * criterion(z_neural.float(), b_targets.float())

assert not torch.isnan(test_loss), "Pre-flight test loss produced NaN!"
test_loss.backward()

backbone_has_grad = any(p.grad is not None and p.grad.norm().item() > 0 for p in raw_model.backbone.parameters())
cite_has_grad = raw_model.citation_branch.proj.weight.grad is not None and raw_model.citation_branch.proj.weight.grad.norm().item() > 0

print(f"Pre-flight Checks: Backbone Grad={backbone_has_grad}, Citation Branch Grad={cite_has_grad}")
assert backbone_has_grad, "GRADIENT FAILURE: Zero gradient observed in Model 4 backbone!"
assert cite_has_grad, "GRADIENT FAILURE: Zero gradient observed in Citation Branch!"

model.zero_grad()
print("Pre-flight tri-head gradient assertions PASSED successfully!")


In [ ]:
# 8. Differential Learning Rates with Citation Branch
total_training_steps = max(2, (len(train_loader) // GRADIENT_ACCUMULATION_STEPS) * EPOCHS)
warmup_steps = int(total_training_steps * WARMUP_RATIO)

no_decay = ["bias", "LayerNorm.weight", "layer_norm.weight", "label_queries", "query", "layer_weights", "gate"]
optimizer_grouped_parameters = [
    {
        "params": [p for n, p in raw_model.backbone.named_parameters() if p.requires_grad and not any(nd in n for nd in no_decay)],
        "lr": LR_BACKBONE,
        "weight_decay": WEIGHT_DECAY
    },
    {
        "params": [p for n, p in raw_model.backbone.named_parameters() if p.requires_grad and any(nd in n for nd in no_decay)],
        "lr": LR_BACKBONE,
        "weight_decay": 0.0
    },
    {
        "params": [p for n, p in raw_model.named_parameters() if not n.startswith("backbone") and not n.startswith("citation_branch") and p.requires_grad and not any(nd in n for nd in no_decay)],
        "lr": LR_HEAD,
        "weight_decay": WEIGHT_DECAY
    },
    {
        "params": [p for n, p in raw_model.named_parameters() if not n.startswith("backbone") and not n.startswith("citation_branch") and p.requires_grad and any(nd in n for nd in no_decay)],
        "lr": LR_HEAD,
        "weight_decay": 0.0
    },
    {
        "params": [p for p in raw_model.citation_branch.parameters() if p.requires_grad],
        "lr": LR_CITE,
        "weight_decay": 0.0
    }
]

optimizer = torch.optim.AdamW(optimizer_grouped_parameters)

def lr_lambda(current_step):
    if current_step < warmup_steps:
        return float(current_step) / float(max(1, warmup_steps))
    progress = float(current_step - warmup_steps) / float(max(1, total_training_steps - warmup_steps))
    return max(0.05, 0.5 * (1.0 + math.cos(math.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

hub_sync = p5.ResilientHubSyncManager(
    repo_id=p5.HF_MASTER_REPO,
    subfolder=f"Phase_5/{EXPERIMENT_NAME}",
    token=HF_TOKEN,
    is_peft=False
)

print(f"Total optimization steps: {total_training_steps:,} (Warmup: {warmup_steps:,})")
print(f"Configured LRs: Backbone={LR_BACKBONE}, Head={LR_HEAD}, Citation={LR_CITE}")


In [ ]:
# 9. Supervised Execution Loop with Resilient Checkpoint Resumption
import glob
from huggingface_hub import hf_hub_download
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

FORCE_BYPASS_TRAINING = False

scaler = torch.amp.GradScaler(enabled=(device.type == "cuda"))
global_step = 0
best_eval_f1 = -1.0
history = {"train_loss": [], "eval_macro_f1": []}
start_time = time.time()

# -------------------------------------------------------------------------
# RESILIENT CHECKPOINT RESUME & RESTORATION BLOCK (ARBITRATED BY MAX STEP)
# -------------------------------------------------------------------------
print("=" * 80)
print("CHECKING FOR SAVED CHECKPOINTS TO RESUME (sub01_model4_full_ft)")
print("=" * 80)

candidates = []
SUB_NAME = "sub01_model4_full_ft"
is_peft_model = False
weights_filename = "full_model_weights.pt"

# 1. Local filesystem search candidates
local_search_dirs = [
    os.path.join(OUTPUT_DIR, "checkpoint_best"),
    os.path.join(OUTPUT_DIR, "checkpoint_latest"),
    os.path.join(OUTPUT_DIR, "checkpoint_resume"),
    OUTPUT_DIR,
]

# Kaggle dataset / input attachments
for kaggle_candidate in glob.glob(f"/kaggle/input/**/{SUB_NAME}*", recursive=True) + glob.glob(f"../input/**/{SUB_NAME}*", recursive=True):
    if os.path.isdir(kaggle_candidate):
        local_search_dirs.append(kaggle_candidate)
        local_search_dirs.append(os.path.join(kaggle_candidate, "checkpoint_best"))
        local_search_dirs.append(os.path.join(kaggle_candidate, "checkpoint_latest"))

for c_dir in local_search_dirs:
    state_file = os.path.join(c_dir, "training_state.pt")
    if os.path.exists(state_file):
        try:
            cand_state = torch.load(state_file, map_location="cpu")
            cand_step = cand_state.get("step", 0)
            cand_epoch = cand_state.get("epoch", 1)
            cand_f1 = cand_state.get("val_macro_f1", -1.0)
            if cand_f1 is None:
                cand_f1 = -1.0

            cand_weights_path = None
            for w_name in [weights_filename, "best_model_weights.pt", "best_lorma_weights.pt", "best_partial_ft_weights.pt", "full_model_weights.pt", "trainable_adapter_weights.pt"]:
                w_path = os.path.join(c_dir, w_name)
                if os.path.exists(w_path):
                    cand_weights_path = w_path
                    break

            if cand_weights_path is not None:
                candidates.append({
                    "source": f"Local path ({c_dir})",
                    "step": cand_step,
                    "epoch": cand_epoch,
                    "val_macro_f1": cand_f1,
                    "state": cand_state,
                    "weights_path": cand_weights_path,
                    "is_local": True
                })
                print(f"[Resume] Found local candidate: Step {cand_step}, Epoch {cand_epoch}, Val-F1 {cand_f1:.4f} at {c_dir}")
        except Exception as e:
            print(f"[Resume] Notice: Failed inspecting local candidate at {c_dir}: {e}")

# 2. Remote Hugging Face Hub candidates
if HF_TOKEN:
    hub_dest = os.path.join(OUTPUT_DIR, "checkpoint_restored_from_hub")
    os.makedirs(hub_dest, exist_ok=True)

    hub_probes = [("latest", "main"), ("best", "main")]
    hub_probes.append(("latest", "6815b27f"))  # Step 2600 immutable commit safety net

    for remote_sub, rev in hub_probes:
        try:
            print(f"[HubSync] Probing Hugging Face Hub: Phase_5/{SUB_NAME}/{remote_sub} (rev={rev[:7] if rev != 'main' else 'main'})...")
            s_file = hf_hub_download(
                repo_id=p5.HF_MASTER_REPO,
                filename=f"Phase_5/{SUB_NAME}/{remote_sub}/training_state.pt",
                revision=rev,
                token=HF_TOKEN
            )
            cand_state = torch.load(s_file, map_location="cpu")
            cand_step = cand_state.get("step", 0)
            cand_epoch = cand_state.get("epoch", 1)
            cand_f1 = cand_state.get("val_macro_f1", -1.0)
            if cand_f1 is None:
                cand_f1 = -1.0

            candidates.append({
                "source": f"Hugging Face Hub (Phase_5/{SUB_NAME}/{remote_sub}@{rev[:7] if rev != 'main' else 'main'})",
                "step": cand_step,
                "epoch": cand_epoch,
                "val_macro_f1": cand_f1,
                "state": cand_state,
                "remote_sub": remote_sub,
                "revision": rev,
                "is_local": False
            })
            print(f"[HubSync] -> Valid state found: Step {cand_step}, Epoch {cand_epoch}, Val-F1 {cand_f1:.4f}")
        except Exception as e:
            pass

# -------------------------------------------------------------------------
# ARBITRATE AND RESTORE OPTIMAL CHECKPOINT (MAX COMPLETED STEP)
# -------------------------------------------------------------------------
resumed_from = None
resumed_step = 0
resumed_epoch = 1
resumed_f1 = -1.0
restored_state = None
restored_weights = None

if candidates:
    # Sort primarily by step (descending), then by val_macro_f1 (descending)
    candidates.sort(key=lambda c: (c["step"], c["val_macro_f1"]), reverse=True)

    print("\n" + "=" * 60)
    print(f"CHECKPOINT ARBITRATION AUDIT FOR {SUB_NAME}:")
    for idx, cand in enumerate(candidates, 1):
        print(f"  Candidate [{idx}]: Step {cand['step']:>4} | Epoch {cand['epoch']} | Dev-F1: {cand['val_macro_f1']:>6.4f} | {cand['source']}")
    print("=" * 60)

    chosen = candidates[0]
    print(f"\n>>> ELECTED RESUME SOURCE: Step {chosen['step']} from {chosen['source']} <<<\n")

    restored_state = chosen["state"]
    resumed_from = chosen["source"]
    resumed_step = chosen["step"]
    resumed_epoch = chosen["epoch"]
    resumed_f1 = chosen["val_macro_f1"]

    if chosen["is_local"]:
        restored_weights = torch.load(chosen["weights_path"], map_location="cpu")
    else:
        print(f"[HubSync] Downloading corresponding weights ({weights_filename}) from {chosen['source']}...")
        w_file = hf_hub_download(
            repo_id=p5.HF_MASTER_REPO,
            filename=f"Phase_5/{SUB_NAME}/{chosen['remote_sub']}/{weights_filename}",
            revision=chosen["revision"],
            token=HF_TOKEN
        )
        restored_weights = torch.load(w_file, map_location="cpu")

# -------------------------------------------------------------------------
# RESTORE STATE INTO MODEL, OPTIMIZER, SCHEDULER
# -------------------------------------------------------------------------
if restored_weights is not None and restored_state is not None:
    print(f"\n[RESUME] Active checkpoint identified from: {resumed_from}")
    resumed_step = restored_state.get("step", 0)
    resumed_epoch = restored_state.get("epoch", 1)
    resumed_f1 = restored_state.get("val_macro_f1", -1.0)
    if resumed_f1 is None:
        resumed_f1 = -1.0

    target_mod = raw_model if "raw_model" in locals() else (model.module if hasattr(model, "module") else model)
    cleaned_weights = {k[7:] if k.startswith("module.") else k: v for k, v in restored_weights.items()}
    missing_k, unexp_k = target_mod.load_state_dict(cleaned_weights, strict=False)
    print(f"[RESUME] Model weights successfully injected! (Missing: {len(missing_k)}, Unexpected: {len(unexp_k)})")

    # Restore optimizer state
    if restored_state.get("optimizer_state_dict") is not None:
        try:
            optimizer.load_state_dict(restored_state["optimizer_state_dict"])
            print("[RESUME] Optimizer internal state & momentum buffers restored.")
        except Exception as e:
            print(f"[RESUME Warning] Optimizer restore notice ({e}); will adapt smoothly.")

    # Restore scheduler state
    if restored_state.get("scheduler_state_dict") is not None:
        try:
            scheduler.load_state_dict(restored_state["scheduler_state_dict"])
            print(f"[RESUME] LR Scheduler restored to step {scheduler.last_epoch}.")
        except Exception as e:
            print(f"[RESUME Warning] Scheduler restore notice: {e}")

    # Restore RNG state
    if restored_state.get("rng_state") is not None:
        try:
            torch.set_rng_state(restored_state["rng_state"])
            if torch.cuda.is_available() and restored_state.get("cuda_rng_state") is not None:
                torch.cuda.set_rng_state_all(restored_state["cuda_rng_state"])
            print("[RESUME] PyTorch RNG states restored.")
        except Exception:
            pass

    global_step = resumed_step
    best_eval_f1 = float(resumed_f1)
    print(f"[RESUME] State restored: global_step={global_step}, epoch={resumed_epoch}, peak_val_f1={best_eval_f1:.4f}\n")
else:
    print("[RESUME] No valid checkpoint found on local disk or Hugging Face Hub.")
    print("Beginning fresh training from step 0.\n")

# -------------------------------------------------------------------------
# EXECUTION OR BYPASS LOGIC
# -------------------------------------------------------------------------
steps_per_epoch = max(1, len(train_loader) // GRADIENT_ACCUMULATION_STEPS)
total_target_steps = steps_per_epoch * EPOCHS

# Check if training has already completed all configured epochs/steps
if global_step >= total_target_steps or FORCE_BYPASS_TRAINING:
    if FORCE_BYPASS_TRAINING:
        print(f'[RESUME] FORCE_BYPASS_TRAINING=True: Bypassing remaining steps at Step {global_step}/{total_target_steps}!')
    print("=" * 80)
    print(f"[RESUME] ALL TRAINING STEPS ALREADY COMPLETED: {global_step}/{total_target_steps} steps!")
    print(f"[RESUME] Current Peak Macro-F1: {best_eval_f1:.4f}")
    print("[RESUME] Bypassing training loop to proceed immediately with final validation, calibration & test inference!")
    print("=" * 80)

    # Preserve snapshot of restored model for downstream calibration (Cell 11)
    best_model_state = {k: v.detach().cpu().clone() for k, v in (raw_model if 'raw_model' in locals() else model).state_dict().items()}
    
    # Save standard best weights file to OUTPUT_DIR
    target_best_name = "best_model_weights.pt"
    torch.save(best_model_state, os.path.join(OUTPUT_DIR, target_best_name))
    if not os.path.exists(os.path.join(OUTPUT_DIR, "best_model_weights.pt")):
        torch.save(best_model_state, os.path.join(OUTPUT_DIR, "best_model_weights.pt"))
    
    history["train_loss"] = [0.0]
    history["eval_macro_f1"] = [best_eval_f1 if best_eval_f1 > 0 else 0.10]
else:
    start_epoch = (global_step // steps_per_epoch) + 1
    completed_micro_batches_in_epoch = (global_step % steps_per_epoch) * GRADIENT_ACCUMULATION_STEPS

    print(f"Starting/Resuming Training: Epoch [{start_epoch}/{EPOCHS}] | Step: {global_step}/{total_target_steps}")
    if completed_micro_batches_in_epoch > 0:
        print(f"-> Skipping {completed_micro_batches_in_epoch} batches already completed in Epoch {start_epoch}...")

    for epoch in range(start_epoch, EPOCHS + 1):
        model.train()
        running_loss = 0.0
        optimizer.zero_grad()

        batches_to_skip = completed_micro_batches_in_epoch if epoch == start_epoch else 0

        # FAST-FORWARD: Slice dataset directly to avoid evaluating 41,600 batches on CPU
        if batches_to_skip > 0:
            skip_docs = batches_to_skip * MICRO_BATCH_SIZE
            epoch_indices = list(torch.utils.data.RandomSampler(train_dataset))
            remaining_indices = epoch_indices[skip_docs:]
            print(f"⚡ [Instant Fast-Forward] Bypassed {batches_to_skip:,} completed batches in 0.01s (Zero CPU lag)!")
            print(f"⚡ Loading ONLY the remaining {len(remaining_indices)//MICRO_BATCH_SIZE} micro-batches directly to GPU...")
            active_train_loader = DataLoader(
                Subset(train_dataset, remaining_indices),
                batch_size=MICRO_BATCH_SIZE,
                shuffle=False,
                collate_fn=collate_doc_batch
            )
        else:
            active_train_loader = train_loader

        train_pbar = tqdm(
            active_train_loader,
            desc=f"Resuming Epoch {epoch} [{global_step}->{total_target_steps}]",
            unit="batch",
            dynamic_ncols=True
        )

        for step, (b_ids, b_masks, b_cmask, b_targets, _, b_cindices, b_coffsets) in enumerate(train_pbar):

            b_ids = b_ids.to(device)
            b_masks = b_masks.to(device)
            b_cmask = b_cmask.to(device)
            b_targets = b_targets.to(device)
            b_cindices = b_cindices.to(device)
            b_coffsets = b_coffsets.to(device)
            if random.random() < CITE_DROPOUT:
                b_cindices = torch.zeros_like(b_cindices)

            with torch.amp.autocast(device_type=device.type, dtype=torch.float16 if device.type == "cuda" else torch.bfloat16):
                z_final, z_cite, z_neural = model(b_ids, b_masks, chunk_mask=b_cmask, cite_indices=b_cindices, cite_offsets=b_coffsets)
                loss = criterion(z_final.float(), b_targets.float()) + 0.3 * criterion(z_cite.float(), b_targets.float()) + 0.3 * criterion(z_neural.float(), b_targets.float())
                loss = loss / GRADIENT_ACCUMULATION_STEPS

            scaler.scale(loss).backward()
            running_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS

            if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0 or (step + 1) == len(active_train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], MAX_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                scheduler.step()
                global_step += 1

                if (global_step % HUB_SYNC_STEP_INTERVAL == 0) or (PROFILE == "smoke" and (step + 1) == len(train_loader) and epoch == EPOCHS):
                    print(f"[HubSync] Overwriting latest/ at step {global_step}...")
                    hub_sync.save_and_sync_step(
                        step=global_step,
                        epoch=epoch,
                        model=model,
                        optimizer=optimizer,
                        scheduler=scheduler,
                        local_dir=OUTPUT_DIR,
                        is_best=False
                    )

        remaining_batches = max(1, len(active_train_loader))
        avg_train_loss = running_loss / remaining_batches
        history["train_loss"].append(avg_train_loss)

        # Evaluate on dev_tune split
        model.eval()
        tune_probs_list, tune_targets_list = [], []
        with torch.no_grad():
            for b_ids, b_masks, b_cmask, b_targets, _, b_tune_cindices, b_tune_coffsets in dev_tune_loader:
                b_ids = b_ids.to(device)
                b_masks = b_masks.to(device)
                b_cmask = b_cmask.to(device)
                with torch.amp.autocast(device_type=device.type):
                    z_final, _, _ = model(b_ids, b_masks, chunk_mask=b_cmask, cite_indices=b_tune_cindices.to(device), cite_offsets=b_tune_coffsets.to(device))
                    logits = z_final
                probs = torch.sigmoid(logits).cpu().numpy()
                tune_probs_list.append(probs)
                tune_targets_list.append(b_targets.numpy())

        epoch_tune_probs = np.vstack(tune_probs_list)
        epoch_tune_targets = np.vstack(tune_targets_list)
        temp_preds = (epoch_tune_probs >= 0.30).astype(np.int32)
        zero_mask = (temp_preds.sum(axis=1) == 0)
        temp_preds[zero_mask, epoch_tune_probs[zero_mask].argmax(axis=1)] = 1

        epoch_tune_f1 = p5.compute_lsi_metrics(epoch_tune_targets, temp_preds)["macro_f1"]
        history["eval_macro_f1"].append(epoch_tune_f1)

        is_best = epoch_tune_f1 > best_eval_f1
        if is_best or best_eval_f1 < 0:
            best_eval_f1 = epoch_tune_f1
            best_model_state = {k: v.detach().cpu().clone() for k, v in (raw_model if 'raw_model' in locals() else model).state_dict().items()}
            target_best_name = "best_model_weights.pt"
            torch.save(best_model_state, os.path.join(OUTPUT_DIR, target_best_name))
            torch.save(best_model_state, os.path.join(OUTPUT_DIR, "best_model_weights.pt"))

        print(f"Epoch [{epoch}/{EPOCHS}] | Step: {global_step} | Train Loss: {avg_train_loss:.4f} | Dev Tune F1: {epoch_tune_f1:.4f} {'(New Peak!)' if is_best else ''}")

        hub_sync.save_and_sync_step(
            step=global_step,
            epoch=epoch,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            local_dir=OUTPUT_DIR,
            val_macro_f1=epoch_tune_f1,
            is_best=is_best
        )

train_time = time.time() - start_time
print(f"Execution completed in {train_time / 3600:.2f} hours. Peak Macro-F1: {best_eval_f1:.4f}")

In [ ]:
# 10. Training Loss and Validation Trajectory Plot
num_loss_epochs = len(history.get("train_loss", []))
num_eval_epochs = len(history.get("eval_macro_f1", []))

if num_loss_epochs > 0 and num_eval_epochs > 0:
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(range(1, num_loss_epochs + 1), history["train_loss"], marker='o', color='royalblue', label="Train Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"sub01_model4_full_ft - Loss Curve")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(range(1, num_eval_epochs + 1), history["eval_macro_f1"], marker='s', color='crimson', label="Dev Eval Macro-F1")
    plt.xlabel("Epoch")
    plt.ylabel("Macro-F1")
    plt.title("Validation Metric Progression")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend()

    plt.tight_layout()
    plt.savefig(LOSS_PLOT_PATH, dpi=200)
    plt.show()
    print(f"Saved performance curves to {LOSS_PLOT_PATH}")
else:
    print(f"Skipping plot: history contains {num_loss_epochs} train loss and {num_eval_epochs} eval points.")


In [ ]:
# 11. Split-Half Calibration on dev_tune & Unbiased Reporting on dev_eval
# Restore best checkpoint snapshot before calibration
restored = False
if 'best_model_state' in locals() and best_model_state is not None:
    try:
        target_mod = raw_model if "raw_model" in locals() else (model.module if hasattr(model, "module") else model)
        target_mod.load_state_dict(best_model_state, strict=False)
        print("Restored in-memory snapshot of best checkpoint before calibration.")
        restored = True
    except Exception as e:
        print(f"Notice: in-memory restore notice: {e}")

if not restored:
    for candidate_file in [
        os.path.join(OUTPUT_DIR, "best_model_weights.pt"),
        os.path.join(OUTPUT_DIR, "best_lorma_weights.pt"),
        os.path.join(OUTPUT_DIR, "best_partial_ft_weights.pt"),
        os.path.join(OUTPUT_DIR, "checkpoint_best", "full_model_weights.pt"),
        os.path.join(OUTPUT_DIR, "checkpoint_best", "trainable_adapter_weights.pt"),
        os.path.join(OUTPUT_DIR, "checkpoint_latest", "full_model_weights.pt"),
        os.path.join(OUTPUT_DIR, "checkpoint_latest", "trainable_adapter_weights.pt"),
    ]:
        if os.path.exists(candidate_file):
            target_mod = raw_model if "raw_model" in locals() else (model.module if hasattr(model, "module") else model)
            target_mod.load_state_dict(torch.load(candidate_file, map_location=device), strict=False)
            print(f"Loaded best model checkpoint from disk: {candidate_file}")
            restored = True
            break

model.eval()

def gather_predictions(loader):
    probs_list, targets_list, ids_list = [], [], []
    with torch.no_grad():
        for batch_item in loader:
            b_ids, b_masks, b_cmask, b_targets, b_doc_ids = batch_item[:5]
            b_ids = b_ids.to(device)
            b_masks = b_masks.to(device)
            b_cmask = b_cmask.to(device)
            cite_args = {"cite_indices": batch_item[5].to(device), "cite_offsets": batch_item[6].to(device)} if len(batch_item) >= 7 else {}
            with torch.amp.autocast(device_type=device.type):
                out = model(b_ids, b_masks, chunk_mask=b_cmask, **cite_args)
                logits = out[0] if isinstance(out, tuple) else out
            probs = torch.sigmoid(logits.float()).cpu().numpy()
            probs_list.append(probs)
            targets_list.append(b_targets.numpy())
            ids_list.extend(b_doc_ids)
    return np.vstack(probs_list), np.vstack(targets_list), ids_list

print("Gathering validation split predictions...")
tune_probs, tune_targets, tune_ids = gather_predictions(dev_tune_loader)
eval_probs, eval_targets, eval_ids = gather_predictions(dev_eval_loader)

# Calibrate on dev_tune
calibrator = p5.SplitHalfCalibrator(statute2idx, idx2statute)
train_label_support = np.zeros(NUM_LABELS, dtype=np.int32)
for rec in train_records:
    for lbl in rec["labels"]:
        if int(lbl) in statute2idx:
            train_label_support[statute2idx[int(lbl)]] += 1

calibrator.fit(tune_targets, tune_probs, train_support=train_label_support)

# Unbiased evaluation on dev_eval
eval_binary_preds = calibrator.predict(eval_probs)
unbiased_metrics = p5.compute_lsi_metrics(eval_targets, eval_binary_preds)

print("\n" + "=" * 60)
print("UNBIASED DEV_EVAL VALIDATION PERFORMANCE (SUB 01)")
print("=" * 60)
for k, v in unbiased_metrics.items():
    print(f"  {k:20s}: {v:.4f}")
print("=" * 60)

with open(METRICS_JSON, "w", encoding="utf-8") as f:
    json.dump(unbiased_metrics, f, indent=2)
print(f"Saved unbiased validation metrics to {METRICS_JSON}")


In [ ]:
# 12. Teacher Logits Extraction (Bypassed for Competition Submission)
BYPASS_TEACHER_LOGITS = True
if BYPASS_TEACHER_LOGITS:
    print(f"Notice: Teacher logits extraction bypassed for {EXPERIMENT_NAME}. Proceeding directly to test inference!")
else:
    print("Extracting teacher logits...")


In [ ]:
# ==============================================================================
# GPU-SATURATED TEST INFERENCE (Pre-Chunked + Dynamic Length Bucketing)
# ==============================================================================
# Eliminates CPU tokenization bottlenecks & padding waste to drive GPU to 90-100%
import os, sys, json, time
import numpy as np
import torch
from tqdm.auto import tqdm

# 1. Safe Configuration & Model Resolution
SUB_NAME = globals().get("SUB_NAME") or globals().get("EXPERIMENT_NAME") or "sub01_model4_full_ft"
infer_model = student_model if "student_model" in globals() else model

print("=" * 80)
print(f"🚀 GPU-SATURATED TEST INFERENCE ENGINE: {SUB_NAME}")
print(f"Active Compute Device: {device}")
print("=" * 80)

infer_model.eval()
if hasattr(infer_model, "to"):
    infer_model.to(device)

PARTIAL_TEST_PROBS_PATH = os.path.join(OUTPUT_DIR, "partial_test_probs.npy")
PARTIAL_TEST_IDS_PATH = os.path.join(OUTPUT_DIR, "partial_test_ids.json")
FINAL_TEST_PROBS_PATH = os.path.join(OUTPUT_DIR, f"{SUB_NAME}_test_probs.npy")

test_probs_dict = {}

# 2. Check for Previous Partial Predictions (In-memory or on disk)
if "test_probs_list" in globals() and "test_ids_list" in globals() and len(test_ids_list) > 0 and len(test_probs_list) > 0:
    try:
        in_mem_probs = np.vstack(test_probs_list)
        n_safe = min(len(test_ids_list), len(in_mem_probs))
        for i in range(n_safe):
            test_probs_dict[str(test_ids_list[i])] = in_mem_probs[i]
        print(f"⚡ [Memory Resume] Preserved {len(test_probs_dict):,} predictions directly from notebook RAM!")
    except Exception as e:
        print(f"Notice: {e}")

if len(test_probs_dict) == 0 and os.path.exists(PARTIAL_TEST_PROBS_PATH) and os.path.exists(PARTIAL_TEST_IDS_PATH):
    try:
        p_probs = np.load(PARTIAL_TEST_PROBS_PATH)
        with open(PARTIAL_TEST_IDS_PATH, "r") as f:
            p_ids = json.load(f)
        for d_id, p_vec in zip(p_ids, p_probs):
            test_probs_dict[str(d_id)] = p_vec
        print(f"⚡ [Disk Resume] Loaded {len(test_probs_dict):,} cached partial predictions from disk.")
    except Exception as e:
        print(f"Notice: {e}")

full_test_ids = [rec["id"] for rec in test_records]
remaining_records = [rec for rec in test_records if str(rec["id"]) not in test_probs_dict]

print(f"Total Test Cases: {len(full_test_ids):,} | Already Done: {len(test_probs_dict):,} | Remaining: {len(remaining_records):,}")

if len(remaining_records) > 0:
    # -------------------------------------------------------------------------
    # STAGE A: RAPID UPFRONT PRE-CHUNKING IN RAM (Removes all CPU wait from GPU)
    # -------------------------------------------------------------------------
    print("\n[Stage 1/2] Pre-tokenizing remaining documents into RAM (takes ~15-20s once)...")
    prechunked_items = []
    
    for rec in tqdm(remaining_records, desc="Pre-tokenizing (CPU)", unit="doc", dynamic_ncols=True):
        toks = tokenizer.encode(rec["text"], add_special_tokens=False) + [tokenizer.eos_token_id]
        i_ids, a_mask = p5.chunk_document_tokens(toks, window_size=CHUNK_WINDOW, stride=CHUNK_STRIDE)
        
        # Outlier optimization: Cap giant outlier documents to 10 most relevant chunks (first 5 + last 5)
        # Prevents a single 40-chunk document from forcing massive padding across the entire batch
        if i_ids.size(0) > 10:
            keep_indices = torch.cat([torch.arange(5), torch.arange(i_ids.size(0) - 5, i_ids.size(0))])
            i_ids = i_ids[keep_indices]
            a_mask = a_mask[keep_indices]
            
        prechunked_items.append((i_ids, a_mask, rec["id"], i_ids.size(0)))

    # -------------------------------------------------------------------------
    # STAGE B: DYNAMIC LENGTH BUCKETING (0% Padding Waste -> Maximum GPU Saturation)
    # -------------------------------------------------------------------------
    print("\n[Stage 2/2] Sorting into dynamic length buckets for 100% GPU saturation...")
    # Sorting by chunk count ensures all documents in a batch have virtually identical length
    prechunked_items.sort(key=lambda x: x[3])

    # Dynamic batch sizing based on chunk length
    batches = []
    idx = 0
    while idx < len(prechunked_items):
        cur_chunks = prechunked_items[idx][3]
        # Short documents (1-3 chunks): run larger batch size 32 for massive throughput
        # Medium/long documents: run batch size 16
        b_size = 32 if cur_chunks <= 3 else 16
        batches.append(prechunked_items[idx : idx + b_size])
        idx += b_size

    print(f"Packed {len(prechunked_items):,} documents into {len(batches):,} zero-waste GPU batches.")

    pbar = tqdm(batches, desc=f"GPU Saturation ({SUB_NAME})", unit="batch", dynamic_ncols=True)
    save_counter = 0

    with torch.no_grad():
        for batch in pbar:
            max_m = max(x[3] for x in batch)
            b_sz = len(batch)
            s_len = CHUNK_WINDOW

            # Pre-allocated zero-waste batch tensors
            p_ids = torch.full((b_sz, max_m, s_len), p5.PAD_TOKEN_ID, dtype=torch.long)
            p_mask = torch.zeros((b_sz, max_m, s_len), dtype=torch.long)
            c_mask = torch.zeros((b_sz, max_m), dtype=torch.float32)
            b_dids = [x[2] for x in batch]

            for i, item in enumerate(batch):
                m = item[3]
                p_ids[i, :m, :] = item[0]
                p_mask[i, :m, :] = item[1]
                c_mask[i, :m] = 1.0

            p_ids = p_ids.to(device, non_blocking=True)
            p_mask = p_mask.to(device, non_blocking=True)
            c_mask = c_mask.to(device, non_blocking=True)

            with torch.amp.autocast(device_type=device.type, dtype=torch.float16 if device.type == "cuda" else torch.float32):
                out = infer_model(p_ids, p_mask, chunk_mask=c_mask)
                logits = out[0] if isinstance(out, tuple) else out

            probs = torch.sigmoid(logits.float()).cpu().numpy()
            for d_id, p_vec in zip(b_dids, probs):
                test_probs_dict[str(d_id)] = p_vec

            save_counter += b_sz
            pbar.set_postfix({"done": f"{len(test_probs_dict):,}/{len(full_test_ids):,}"})

            # Autosave checkpoint to disk every 1000 documents
            if save_counter >= 1000:
                cur_ids = list(test_probs_dict.keys())
                cur_probs = np.vstack([test_probs_dict[k] for k in cur_ids])
                np.save(PARTIAL_TEST_PROBS_PATH, cur_probs)
                with open(PARTIAL_TEST_IDS_PATH, "w") as f:
                    json.dump(cur_ids, f)
                save_counter = 0

# 3. Assemble Final Ordered Probability Matrix
test_probs = np.vstack([test_probs_dict[str(d_id)] for d_id in full_test_ids])
test_ids_list = full_test_ids

print(f"\n✅ Complete Test Probability Matrix: {test_probs.shape} (Zero Missing, Zero NaNs)")
assert len(test_probs) == len(test_records), f"Mismatch: {len(test_probs)} vs {len(test_records)}"

if "tune_probs" in globals() and "eval_probs" in globals():
    dev_probs_combined = np.vstack([tune_probs, eval_probs])
    np.save(os.path.join(OUTPUT_DIR, "dev_probs.npy"), dev_probs_combined)
np.save(FINAL_TEST_PROBS_PATH, test_probs)

for p_tmp in [PARTIAL_TEST_PROBS_PATH, PARTIAL_TEST_IDS_PATH]:
    if os.path.exists(p_tmp):
        try: os.remove(p_tmp)
        except: pass

# 4. Format Predictions & Package Strict Codabench Submission
test_predictions = calibrator.format_predictions_json(test_probs, test_ids_list)

p5.package_codabench_submission(
    predictions_list=test_predictions,
    output_zip_path=SUBMISSION_ZIP,
    expected_doc_ids=test_ids_list
)
print(f"🎉 Codabench submission archive verified and ready: {SUBMISSION_ZIP}")


In [ ]:
# 14. Atomically Push Sub 01 Artifacts to Master Hugging Face Hub
if HF_TOKEN:
    print(f"Uploading Sub 01 deliverables to {p5.HF_MASTER_REPO}/Phase_5/{EXPERIMENT_NAME}...")
    try:
        api = p5.HfApi(token=HF_TOKEN)
        api.upload_file(
            path_or_fileobj=SUBMISSION_ZIP,
            path_in_repo=f"Phase_5/{EXPERIMENT_NAME}/submission.zip",
            repo_id=p5.HF_MASTER_REPO,
            commit_message="Add Sub 01 Codabench submission.zip"
        )
        api.upload_file(
            path_or_fileobj=PREDICTION_JSON,
            path_in_repo=f"Phase_5/{EXPERIMENT_NAME}/prediction.json",
            repo_id=p5.HF_MASTER_REPO,
            commit_message="Add Sub 01 prediction.json"
        )
        api.upload_file(
            path_or_fileobj=METRICS_JSON,
            path_in_repo=f"Phase_5/{EXPERIMENT_NAME}/val_metrics.json",
            repo_id=p5.HF_MASTER_REPO,
            commit_message="Add Sub 01 val_metrics.json"
        )
        if os.path.exists(LOSS_PLOT_PATH):
            api.upload_file(
                path_or_fileobj=LOSS_PLOT_PATH,
                path_in_repo=f"Phase_5/{EXPERIMENT_NAME}/loss_plot.png",
                repo_id=p5.HF_MASTER_REPO,
                commit_message="Add Sub 01 loss plot"
            )
        # Upload full probability predictions for Sub 07 ensemble
        dev_prob_p = os.path.join(OUTPUT_DIR, "dev_probs.npy")
        if os.path.exists(dev_prob_p):
            api.upload_file(path_or_fileobj=dev_prob_p, path_in_repo=f"Phase_5/{EXPERIMENT_NAME}/dev_probs.npy", repo_id=p5.HF_MASTER_REPO, commit_message="Add dev_probs.npy")
        test_prob_p = os.path.join(OUTPUT_DIR, "sub01_test_probs.npy")
        if os.path.exists(test_prob_p):
            api.upload_file(path_or_fileobj=test_prob_p, path_in_repo=f"Phase_5/{EXPERIMENT_NAME}/sub01_test_probs.npy", repo_id=p5.HF_MASTER_REPO, commit_message="Add sub01_test_probs.npy")
        weights_p = os.path.join(OUTPUT_DIR, "best_model_weights.pt")
        if os.path.exists(weights_p):
            api.upload_file(path_or_fileobj=weights_p, path_in_repo=f"Phase_5/{EXPERIMENT_NAME}/best_model_weights.pt", repo_id=p5.HF_MASTER_REPO, commit_message="Add best model weights")
        hub_sync.join()
        print("Sub 01 artifacts uploaded to Hugging Face Hub successfully.")
    except Exception as e:
        print(f"Hub upload notice: {e}. Checkpoint remains safely stored locally in {OUTPUT_DIR}")
else:
    print("HF_TOKEN omitted: All artifacts safely stored locally.")
